 修改dataset,改进数据整除
 跑VQF：baseline

In [ ]:
 
为现有的 KalmanNet 使用 GRU 学习卡尔曼增益网络（KGainNet）改进为自适应网络，可以从以下几个关键方面进行增强设计。这些改进可以增强网络对动态环境变化的适应能力，提高噪声估计的准确性和状态估计的鲁棒性。

---

### **改进建议与实现**

#### **1. 动态估计噪声协方差矩阵 $Q$ 和 $R$**
   - **改进目标**：
     动态调整过程噪声协方差 $Q$ 和观测噪声协方差 $R$ 以适应环境变化。
   - **方法**：
     - 增加两个独立的 GRU 模块分别用于估计 $Q$ 和 $R$，根据当前观测和历史状态动态更新噪声协方差。
     - 在损失函数中加入对 $Q$ 和 $R$ 估计的正则化项，确保矩阵的正定性。

   **代码实现**：
   ```python
   # GRU for dynamic Q estimation
   self.d_input_Q_adaptive = self.m * args.in_mult_KNet
   self.d_hidden_Q_adaptive = self.m ** 2
   self.GRU_Q_adaptive = nn.GRU(self.d_input_Q_adaptive, self.d_hidden_Q_adaptive).to(self.device)

   # GRU for dynamic R estimation
   self.d_input_R_adaptive = self.n * args.in_mult_KNet
   self.d_hidden_R_adaptive = self.n ** 2
   self.GRU_R_adaptive = nn.GRU(self.d_input_R_adaptive, self.d_hidden_R_adaptive).to(self.device)
   ```

---

#### **2. 增强非线性特性建模**
   - **改进目标**：
     增强系统对非线性状态转移和观测模型的适应能力。
   - **方法**：
     - 使用更深的神经网络或注意力机制（Transformer）代替部分 GRU 模块，用于捕捉非线性关系。
     - 在 GRU 的输入中加入非线性特征提取层（如 MLP）。

   **代码实现**：
   ```python
   # Nonlinear transformation for GRU inputs
   self.NonLinear_Q = nn.Sequential(
       nn.Linear(self.d_input_Q, 128),
       nn.ReLU(),
       nn.Linear(128, self.d_input_Q)
   ).to(self.device)

   self.NonLinear_R = nn.Sequential(
       nn.Linear(self.d_input_R, 128),
       nn.ReLU(),
       nn.Linear(128, self.d_input_R)
   ).to(self.device)

   # Apply transformation in forward pass
   Q_input_transformed = self.NonLinear_Q(Q_input)
   R_input_transformed = self.NonLinear_R(R_input)
   ```

---

#### **3. 自适应学习卡尔曼增益**
   - **改进目标**：
     动态调整卡尔曼增益 $K_t$ 的计算规则，避免依赖固定公式。
   - **方法**：
     - 用 GRU 模块学习增益更新规则，结合前一时间步的状态协方差 $P_{t-1}$ 和当前观测协方差 $R_t$ 生成增益。
     - 增益学习部分可以通过 MLP 输出动态调整的卡尔曼增益。

   **代码实现**：
   ```python
   # GRU for learning Kalman Gain K_t
   self.d_input_K = self.m ** 2 + self.n ** 2
   self.d_hidden_K = self.m * self.n
   self.GRU_K = nn.GRU(self.d_input_K, self.d_hidden_K).to(self.device)

   # Fully connected layer to refine Kalman Gain
   self.FC_K = nn.Sequential(
       nn.Linear(self.d_hidden_K, self.m * self.n),
       nn.ReLU(),
       nn.Linear(self.m * self.n, self.m * self.n)
   ).to(self.device)
   ```

---

#### **4. 引入时间注意力机制**
   - **改进目标**：
     在时间维度上动态加权不同时间步的历史状态，适应复杂动态系统。
   - **方法**：
     - 增加时间注意力机制（如 Transformer Encoder 或自注意力机制）处理历史状态序列。
     - 将 GRU 的隐状态作为时间注意力输入，动态权重更新。

   **代码实现**：
   ```python
   # Self-Attention mechanism for temporal modeling
   self.attention = nn.MultiheadAttention(embed_dim=self.d_hidden_Sigma, num_heads=4).to(self.device)

   def forward(self, x_seq):
       # x_seq: [seq_len, batch_size, feature_dim]
       attn_output, _ = self.attention(x_seq, x_seq, x_seq)
       return attn_output
   ```

---

#### **5. 噪声自适应约束**
   - **改进目标**：
     降低噪声对系统的干扰，提高网络的鲁棒性。
   - **方法**：
     - 在损失函数中加入噪声动态估计的正则化项。
     - 动态调整 $Q$ 和 $R$ 时，增加噪声特性（如正定性和稀疏性）的约束。

   **代码实现**：
   ```python
   def loss_function(self, predicted, target, Q, R):
       estimation_loss = nn.MSELoss()(predicted, target)
       Q_regularization = torch.norm(Q - torch.diag(torch.diagonal(Q)))  # 对角化约束
       R_regularization = torch.norm(R - torch.diag(torch.diagonal(R)))
       return estimation_loss + 0.1 * (Q_regularization + R_regularization)
   ```

---

#### **6. 融入外部信息作为上下文特征**
   - **改进目标**：
     结合额外的传感器信息或环境数据，进一步提高状态估计的准确性。
   - **方法**：
     - 在 GRU 输入中增加外部上下文特征（如 GPS、环境温度），提升网络的全局感知能力。
     - 上下文特征可以通过嵌入网络进行预处理。

   **代码实现**：
   ```python
   self.context_embedding = nn.Linear(context_dim, self.m * args.in_mult_KNet).to(self.device)

   def forward(self, x, context):
       context_features = self.context_embedding(context)
       x_with_context = torch.cat((x, context_features), dim=-1)
       return x_with_context
   ```

---

### **改进后的架构优势**

1. **动态噪声估计**：
   - 自适应调整过程噪声 $Q$ 和观测噪声 $R$，增强鲁棒性。
   
2. **非线性建模**：
   - 引入深度神经网络或注意力机制，适应非线性状态转移和观测。

3. **时间序列优化**：
   - 利用时间注意力机制和 GRU 的结合，更好地捕捉动态系统的时间依赖性。

4. **上下文增强**：
   - 融入外部信息，提高网络在复杂环境下的表现。

---

### **总结**

通过增加动态噪声协方差估计、非线性建模、自适应卡尔曼增益和时间注意力机制，可以显著增强 KalmanNet 的自适应能力。改进后的网络适合处理非线性、非平稳系统，以及在动态噪声环境中提供更稳定和准确的状态估计。

In [ ]:
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from scipy.io import loadmat

import os
import torch
from torch.utils.data import Dataset
from scipy.io import loadmat
import numpy as np

class BroadDataset(Dataset):
    def __init__(self, data_dir, validation_files=None, test_files=None):
        """
        初始化 BroadDataset 数据集。
        Args:
            data_dir (str): 数据文件夹路径，包含所有的 .mat 文件。
            validation_files (list): 验证集文件列表。
            test_files (list): 测试集文件列表。
        """
        self.data_dir = data_dir
        self.validation_files = validation_files if validation_files else []
        self.test_files = test_files if test_files else []
        self.file_paths = self._get_all_files(data_dir)  # 获取文件夹中所有 .mat 文件的路径
        self.data = []
        self.load_data()

    def _get_all_files(self, data_dir):
        """
        获取文件夹中所有的 .mat 文件路径。
        Args
            data_dir (str): 文件夹路径。
        Returns:
            list: 包含所有 .mat 文件路径的列表。
        """
        return [os.path.join(data_dir, f) for f in os.listdir(data_dir) if f.endswith('.mat')]

    def load_data(self):
        """
        加载文件夹中的所有 .mat 文件数据，并对每个文件进行数据清理。
        """
        for file_path in self.file_paths:
            file_name = os.path.basename(file_path)
            if file_name in self.validation_files or file_name in self.test_files:
                continue  # 排除验证集和测试集文件

            mat_contents = loadmat(file_path)  # 加载单个 .mat 文件
            print(f"Processing {file_name}")

            raw_imu_acc = torch.tensor(mat_contents['imu_acc'], dtype=torch.float32)
            raw_imu_gyr = torch.tensor(mat_contents['imu_gyr'], dtype=torch.float32)
            raw_imu_mag = torch.tensor(mat_contents['imu_mag'], dtype=torch.float32)
            raw_opt_quat = torch.tensor(mat_contents['opt_quat'], dtype=torch.float32)

            # 清理数据，处理 NaN 值
            imu_acc, imu_gyr, imu_mag, opt_quat = self.clean_fix_data(
                raw_imu_acc, raw_imu_gyr, raw_imu_mag, raw_opt_quat
            )

            if imu_acc is None or imu_gyr is None or imu_mag is None or opt_quat is None:
                print(f"Skipping {file_name} due to missing data")
                continue

            N = imu_acc.shape[0]
            if N != imu_gyr.shape[0] or N != imu_mag.shape[0] or N != opt_quat.shape[0]:
                print(f"Skipping {file_name} due to inconsistent data lengths")
                continue

            # 保存处理后的数据
            self.data.append({
                'gyro': imu_gyr,
                'acc': imu_acc,
                'mag': imu_mag,
                'quaternion': opt_quat
            })

    ### input_nan fix
    def slerp(self, q0, q1, taus):
        """批量进行四元数插值操作"""
        dot = (q0 * q1).sum(dim=1)
        dot = torch.clamp(dot, -1.0, 1.0)  # 使用 torch.clamp 代替手动限制

        theta_0 = torch.acos(dot)
        sin_theta_0 = torch.sin(theta_0)

        theta = theta_0.unsqueeze(-1) * taus
        sin_theta = torch.sin(theta)

        s0 = torch.cos(theta_0.unsqueeze(-1) - theta) / (sin_theta_0.unsqueeze(-1) + 1e-7)
        s1 = sin_theta / (sin_theta_0.unsqueeze(-1) + 1e-7)

        q_interp = s0 * q0.unsqueeze(1) + s1 * q1.unsqueeze(1)
        q_interp = q_interp / q_interp.norm(dim=-1, keepdim=True)  # 单位化

        return q_interp

    def interpolate_quaternions(self, input_quat):
        """快速插值四元数，使用批量处理代替循环"""
        nan_mask = torch.isnan(input_quat[:, 0])
        valid_indices = torch.nonzero(~nan_mask, as_tuple=False).squeeze()
        
        if len(valid_indices) < 2:
            raise ValueError("Not enough valid quaternions to interpolate.")
        
        output_quat = input_quat.clone()
        
        # 计算所有插值间隔
        start_indices = valid_indices[:-1]
        end_indices = valid_indices[1:]

        for start_idx, end_idx in zip(start_indices, end_indices):
            if start_idx >= end_idx - 1:
                continue

            q0 = input_quat[start_idx]
            q1 = input_quat[end_idx]

            # 批量生成 tau (0 到 1)
            nan_indices = torch.arange(start_idx + 1, end_idx)
            assert end_idx != start_idx,"end_idx cannot equal start_idx"
            taus = (nan_indices - start_idx).float() / (end_idx - start_idx)
            
            # 批量插值四元数
            interpolated_quats = self.slerp(q0.unsqueeze(0), q1.unsqueeze(0), taus.unsqueeze(1))
            output_quat[nan_indices] = interpolated_quats.squeeze(1)

        return output_quat

    def clean_fix_data(self, input_acc, input_gyr, input_mag, input_quat_nan):
        """清理并修复加速度计、陀螺仪、磁力计和四元数数据"""
        quat_nan_mask = torch.isnan(input_quat_nan[:, 0])

        first_valid_idx = torch.nonzero(~quat_nan_mask, as_tuple=False)[0].item()
        last_valid_idx = torch.nonzero(~quat_nan_mask, as_tuple=False)[-1].item()

        cleaned_quat = input_quat_nan[first_valid_idx:last_valid_idx + 1]
        fixed_quat = self.interpolate_quaternions(cleaned_quat)
        cleaned_acc = input_acc[first_valid_idx:last_valid_idx + 1]
        cleaned_gyr = input_gyr[first_valid_idx:last_valid_idx + 1]
        cleaned_mag = input_mag[first_valid_idx:last_valid_idx + 1]

        if torch.any(torch.isnan(fixed_quat)):
            print("opt_quat 中仍存在 NaN 值")

        return cleaned_acc, cleaned_gyr, cleaned_mag, fixed_quat


    def __getitem__(self, idx):
        """
        按索引获取数据。
        Args:
            idx (int): 数据索引。
        Returns:
            tuple: 包含 gyro, acc, mag, quaternion 的元组。
        """
        # 找到文件中对应序列的具体索引
        file_idx = 0  # 当前文件索引
        local_idx = idx  # 数据在当前文件中的索引

        # 确定 idx 对应哪个文件的数据（因为多个文件数据是连接的）
        for i, file_data in enumerate(self.data):
            if local_idx < len(file_data['gyro']):
                file_idx = i
                break
            else:
                local_idx -= len(file_data['gyro'])

        # 从对应文件中取出指定索引的序列
        gyro = torch.tensor(self.data[file_idx]['gyro'][local_idx], dtype=torch.float32)
        acc = torch.tensor(self.data[file_idx]['acc'][local_idx], dtype=torch.float32)
        mag = torch.tensor(self.data[file_idx]['mag'][local_idx], dtype=torch.float32)
        quaternion = torch.tensor(self.data[file_idx]['quaternion'][local_idx], dtype=torch.float32)

        return gyro, acc, mag, quaternion

    def __len__(self):
        """
        返回数据集的总长度（所有文件中的序列总和）。
        """
        total_length = 0
        for file_data in self.data:
            total_length += len(file_data['gyro'])  # 每个文件中的 gyro 数量等于序列数量
        return total_length


import os
import torch
from torch.utils.data import Dataset
from scipy.io import loadmat

class BroadDataset(Dataset):
    def __init__(self, data_dir):
        """
        初始化 BroadDataset 数据集。
        Args:
            data_dir (str): 数据文件夹路径，包含所有的 .mat 文件。
        """
        self.data_dir = data_dir
        self.file_paths = self._get_all_files(data_dir)  # 获取文件夹中所有 .mat 文件的路径
        self.data = []
        self.load_data()

    def _get_all_files(self, data_dir):
        """
        获取文件夹中所有的 .mat 文件路径。
        Args:
            data_dir (str): 文件夹路径。
        Returns:
            list: 包含所有 .mat 文件路径的列表。
        """
        return [os.path.join(data_dir, f) for f in os.listdir(data_dir) if f.endswith('.mat')]

    def load_data(self):
        """
        加载文件夹中的所有 .mat 文件数据。
        将每个文件的 imu 和 opt 数据添加到 self.data 列表中。
        """
        for file_path in self.file_paths:
            mat_contents = loadmat(file_path)  # 加载单个 .mat 文件
            imu_gyr = torch.tensor(mat_contents['imu_gyr'])
            imu_acc = torch.tensor(mat_contents['imu_acc'])
            imu_mag = torch.tensor(mat_contents['imu_mag'])
            opt_quat = torch.tensor(mat_contents['opt_quat'])
            opt_pos = torch.tensor(mat_contents['opt_pos'])

            # 保存每个文件的 imu 和 opt 数据到 self.data 列表
            self.data.append({
                'gyro': imu_gyr,
                'acc': imu_acc,
                'mag': imu_mag,
                'quaternion': opt_quat,
                'position': opt_pos
            })

    def __getitem__(self, idx):
        """
        按索引获取数据。
        Args:
            idx (int): 数据索引。
        Returns:
            tuple: 包含 gyro, acc, mag, quaternion, position 的元组。
        """
        # 找到文件中对应序列的具体索引
        file_idx = 0  # 当前文件索引
        local_idx = idx  # 数据在当前文件中的索引

        # 确定 idx 对应哪个文件的数据（因为多个文件数据是连接的）
        for i, file_data in enumerate(self.data):
            if local_idx < len(file_data['gyro']):
                file_idx = i
                break
            else:
                local_idx -= len(file_data['gyro'])

        # 从对应文件中取出指定索引的序列
        gyro = torch.tensor(self.data[file_idx]['gyro'][local_idx], dtype=torch.float32)
        acc = torch.tensor(self.data[file_idx]['acc'][local_idx], dtype=torch.float32)
        mag = torch.tensor(self.data[file_idx]['mag'][local_idx], dtype=torch.float32)
        quaternion = torch.tensor(self.data[file_idx]['quaternion'][local_idx], dtype=torch.float32)
        position = torch.tensor(self.data[file_idx]['position'][local_idx], dtype=torch.float32)

        return gyro, acc, mag, quaternion, position

    def __len__(self):
        """
        返回数据集的总长度（所有文件中的序列总和）。
        Returns:
            int: 数据集总长度。
        """
        total_length = 0
        for file_data in self.data:
            total_length += len(file_data['gyro'])  # 每个文件中的 gyro 数量等于序列数量
        return total_length
